# AutoDrive

## I. Getting started guide

### 1. AutoDrive model introduction

AutoDrive is an **end-to-end** model that maps **two consecutive** front-camera crops (previous frame + current frame) to three outputs:

- **Distance** — normalised value in \([0,1]\) converted to metres with a 150 m cap (see `Models/data_utils/load_data_auto_drive.py`).
- **Road curvature** — in \(1/\mathrm{m}\); training uses a fixed scale `CURV_SCALE` so the head stays well-conditioned.
- **CIPO flag** — raw logit; apply **sigmoid** for probability (a common validation threshold is **0.65**).

Architecture sketch:

- **Backbone** — same family as **AutoSpeed** (shared design); can be initialised from `autospeed.pt`.
- **Head** — fuses high-level features from both frames and regresses distance + curvature + flag.

The default training pipeline applies a **50° horizontal field-of-view centre crop** from the full fisheye frame, then resizes to **1024 × 512** and applies **ImageNet** normalisation. Match that at inference for best results.

### 2. Environment setup

**If you already completed environment setup in another E2E model tutorial in this repo, skip to `3. Download models`.**

Clone this repository (or your organisation fork) if you have not already. Adjust `REPO_DIR` in the next cell to match your local path.

In [ ]:
import os

# Example: clone Autoware POV / VisionPilot repo (update URL to your fork if needed)
REPO_DIR = os.path.abspath("./autoware.privately-owned-vehicles")
if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/autowarefoundation/autoware.privately-owned-vehicles.git

Install Python dependencies from the repository `Models/requirements.txt` (PyTorch, torchvision, PIL, etc.).

In [ ]:
%pip install --upgrade pip
%pip install -r autoware.privately-owned-vehicles/Models/requirements.txt

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

### 3. Download models

**Release weights (placeholder)** — place the published **`AutoDrive.pt`** (or your training **`AutoDrive_best.pth`**) under `./weights/` next to this notebook.

When the final public link is ready, add it here and optionally wire `gdown` like the AutoSpeed tutorial.

**Backbone warm-start (optional)** — for training from scratch with a good backbone, keep **`autospeed.pt`** (AutoSpeed 2.0, 1024×512) available; `AutoDrive.load_backbone_from_autospeed()` copies matching weights.

In [ ]:
import os

weights_dir = os.path.join(os.getcwd(), "weights")
os.makedirs(weights_dir, exist_ok=True)

AUTODRIVE_PT = os.path.join(weights_dir, "AutoDrive.pt")
AUTODRIVE_FALLBACK = os.path.join(weights_dir, "AutoDrive_best.pth")

if os.path.isfile(AUTODRIVE_PT):
    print(f"Found release weights: {AUTODRIVE_PT}")
elif os.path.isfile(AUTODRIVE_FALLBACK):
    print(f"Using training checkpoint: {AUTODRIVE_FALLBACK}")
else:
    print(
        "No weights yet. Copy AutoDrive.pt (release) or AutoDrive_best.pth into ./weights/ "
        "or update this cell with gdown once the download ID is published."
    )

## II. Quick test — single forward pass

Prerequisites: completed **Environment setup** and placed weights under `./weights/` **or** point `WEIGHTS_PATH` to any `AutoDrive_*.pth` from `training/autodrive/run*/checkpoints/`.

AutoDrive always needs **two** RGB tensors `(1, 3, 512, 1024)` — previous and current — in the same normalised space as training.

In [ ]:
import math
import sys
from pathlib import Path

import torch
import torchvision.transforms.functional as TF
from PIL import Image

# --- Locate repository root (directory that contains Models/training/train_auto_drive.py) ---
ROOT = Path(".").resolve()
for _ in range(8):
    if (ROOT / "Models" / "training" / "train_auto_drive.py").is_file():
        break
    ROOT = ROOT.parent
else:
    raise RuntimeError("Could not find repo root. Open the notebook from inside the repo or set ROOT manually.")

sys.path.insert(0, str(ROOT))

from Models.model_components.autodrive.autodrive_network import AutoDrive
from Models.data_utils.load_data_auto_drive import CURV_SCALE, _center_crop_50deg_resize, _read_hfov_deg

_IMAGENET_MEAN = [0.485, 0.456, 0.406]
_IMAGENET_STD = [0.229, 0.224, 0.225]
_D_MAX = 150.0
_WHEELBASE_M = 2.984
_STEER_RATIO = 16.8


def curvature_to_steer_deg(c_1_per_m: float) -> float:
    return math.degrees(math.atan(c_1_per_m * _WHEELBASE_M) * _STEER_RATIO)


def to_model_tensor(np_rgb_hwc_uint8, device):
    t = TF.to_tensor(np_rgb_hwc_uint8)  # CHW float 0..1
    t = TF.normalize(t, _IMAGENET_MEAN, _IMAGENET_STD)
    return t.unsqueeze(0).to(device)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

weights_dir = Path("./weights")
WEIGHTS_PATH = weights_dir / "AutoDrive.pt"
if not WEIGHTS_PATH.is_file():
    WEIGHTS_PATH = weights_dir / "AutoDrive_best.pth"
if not WEIGHTS_PATH.is_file():
    # Example: point at a run you already have under ZOD/training
    WEIGHTS_PATH = ROOT / "Models" / ".."  # user should edit
    print("Edit WEIGHTS_PATH in this cell to your checkpoint, e.g. ~/zod/training/autodrive/run002/checkpoints/AutoDrive_best.pth")

# --- Optional: two real frames from ZOD (set ZOD_ROOT and SEQ if available) ---
ZOD_ROOT = Path.home() / "Downloads" / "data" / "zod"  # change if needed
SEQ = "001451"

model = AutoDrive().to(device)
if WEIGHTS_PATH.is_file():
    ckpt = torch.load(str(WEIGHTS_PATH), map_location=device, weights_only=False)
    sd = ckpt["model"] if isinstance(ckpt, dict) and "model" in ckpt else ckpt
    missing, unexpected = model.load_state_dict(sd, strict=False)
    if missing:
        print("missing keys (first 5):", list(missing)[:5])
    if unexpected:
        print("unexpected keys (first 5):", list(unexpected)[:5])
else:
    print("Skipping weight load — model is randomly initialised for shape smoke test")

model.eval()

label_dir = ZOD_ROOT / "labels" / SEQ
if label_dir.is_dir():
    import json
    from Models.data_parsing.AutoDrive.zod.zod_utils import get_images_blur_dir

    jsons = sorted(label_dir.glob("*.json"))
    if len(jsons) >= 2:
        hfov = _read_hfov_deg(ZOD_ROOT, SEQ)
        img_dir = get_images_blur_dir(ZOD_ROOT, SEQ)
        paths = []
        for lf in jsons[:2]:
            with open(lf) as fh:
                rec = json.load(fh)
            paths.append(img_dir / rec["image"])
        prev_np = _center_crop_50deg_resize(Image.open(paths[0]).convert("RGB"), hfov)
        curr_np = _center_crop_50deg_resize(Image.open(paths[1]).convert("RGB"), hfov)
        t_prev = to_model_tensor(prev_np, device)
        t_curr = to_model_tensor(curr_np, device)
    else:
        t_prev = torch.zeros(1, 3, 512, 1024, device=device)
        t_curr = torch.zeros(1, 3, 512, 1024, device=device)
else:
    t_prev = torch.zeros(1, 3, 512, 1024, device=device)
    t_curr = torch.zeros(1, 3, 512, 1024, device=device)

with torch.no_grad():
    d_norm, curv_norm, flag_logit = model(t_prev, t_curr)

d_norm_v = float(d_norm.squeeze().cpu())
curv_v = float(curv_norm.squeeze().cpu()) * CURV_SCALE
prob = float(torch.sigmoid(flag_logit.squeeze()).cpu())

dist_m = _D_MAX * (1.0 - max(0.0, min(1.0, d_norm_v)))
steer_deg = curvature_to_steer_deg(curv_v)

print(f"d_norm (raw head): {d_norm_v:.4f}")
print(f"distance (m, capped): {dist_m:.1f}")
print(f"curvature (1/m): {curv_v:.5f}")
print(f"steering proxy (deg): {steer_deg:.2f}")
print(f"CIPO probability: {prob:.3f}")

## III. Training and batch inference

### 1. Train (or fine-tune) AutoDrive

Entry script: **`Models/training/train_auto_drive.py`**.

Typical ingredients:

- ZOD root with `labels/` and image shards.
- Optional `--autospeed_ckpt` pointing at **`autospeed.pt`** to initialise the backbone.
- Checkpoints written under `{zod_root}/training/autodrive/run*/checkpoints/`.

Run from the **repository root**:

```bash
python Models/training/train_auto_drive.py --help
```

### 2. Videos and benchmarks

| Script | Purpose |
|--------|---------|
| `Models/inference/autodrive_curvature_video.py` | Overlay distance / curvature / CIPO on video |
| `Models/inference/comparison_video.py` | Classical (AutoSpeed + ObjectFinder) vs AutoDrive stacked view |
| `Models/inference/classical_vs_autodrive_benchmark.py` | Metrics on fixed sequence list |
| `Models/inference/autodrive_benchmark.py` | Two-checkpoint AutoDrive-only benchmark |

Run examples from repo root; pass `--root` to your ZOD dataset root.